In [ ]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim

The dataset I have used in this example is the WI Breast Cancer diagnostic data. More info is available here: https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic. It is actually built-in to sk-learn so I've used that library to load the data below.

In [ ]:
# Loading the data and some preprocessing:
wdbc = load_breast_cancer(as_frame=True)
X, y = wdbc.data, wdbc.target

# Scale data (crucial for neural networks)
scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X).astype(np.float32))
y = pd.DataFrame(y.astype(np.float32))

# train / test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=499)

# train / val split
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=499)

display(X_train.head())
display(y_train.head())

The following code block defines 2 model classes: one with dropout and one without.

In [ ]:
class FCNN(nn.Module):
  def __init__(self, input_size):
    super(FCNN, self).__init__()
    self.fc1 = nn.Linear(input_size, 32)
    self.fc2 = nn.Linear(32, 32)
    self.fc3 = nn.Linear(32, 1)
    self.sigmoid = nn.Sigmoid()
    self.relu = nn.ReLU()

  def forward(self, x):
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)
    x = self.relu(x)
    x = self.fc3(x)
    return self.sigmoid(x)

class FCNNreg(nn.Module):
  def __init__(self, input_size):
    super(FCNNreg, self).__init__()
    self.fc1 = nn.Linear(input_size, 32)
    self.fc2 = nn.Linear(32, 32)
    self.fc3 = nn.Linear(32, 1)
    self.dropout = nn.Dropout(0.2)
    self.sigmoid = nn.Sigmoid()
    self.relu = nn.ReLU()

  def forward(self, x):
    x = self.fc1(x)
    x = self.relu(x)
    x = self.dropout(x)
    x = self.fc2(x)
    x = self.relu(x)
    x = self.dropout(x)
    x = self.fc3(x)
    return self.sigmoid(x)

In [ ]:
# To use PyTorch you must convert our data to PyTorch tensors
X_train_torch = torch.tensor(X_train.values, dtype=torch.float32)
y_train_torch = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_val_torch = torch.tensor(X_val.values, dtype=torch.float32)
y_val_torch = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)

# Initialize the models we defined above
model = FCNN(input_size=X_train.shape[1])
model_drop = FCNNreg(input_size=X_train.shape[1])
model_weight_decay = FCNN(input_size=X_train.shape[1])

# Define our Loss function and algorithm for optimization (aka our 'Optimizer')
criterion = nn.BCELoss()
model_optimizer = optim.SGD(model.parameters(), lr=0.005)
model_drop_optimizer = optim.SGD(model_drop.parameters(), lr=0.005)
model_weight_decay_optimizer = optim.SGD(model_weight_decay.parameters(), lr=0.01, weight_decay=0.005)

# Dictionaries for Loss Values
train_losses = {'model': [], 'dropout': [], 'weight_decay': []}
val_losses = {'model': [], 'dropout': [], 'weight_decay': []}

# training loop
epochs = 50_000

for epoch in range(epochs):

    model.train()
    model_drop.train()
    model_weight_decay.train()

    # forward pass
    model_y_pred = model(X_train_torch)
    model_drop_y_pred = model_drop(X_train_torch)
    model_weight_decay_y_pred = model_weight_decay(X_train_torch)

    # calculate the cost not the loss but all PyTorch syntax calls this the loss
    model_loss = criterion(model_y_pred, y_train_torch)
    model_drop_loss = criterion(model_drop_y_pred, y_train_torch)
    model_weight_decay_loss = criterion(model_weight_decay_y_pred, y_train_torch)
    train_losses['model'].append(model_loss.item())
    train_losses['dropout'].append(model_drop_loss.item())
    train_losses['weight_decay'].append(model_weight_decay_loss.item())

    # backprop
    model_optimizer.zero_grad()
    model_drop_optimizer.zero_grad()
    model_weight_decay_optimizer.zero_grad()
    model_loss.backward()
    model_drop_loss.backward()
    model_weight_decay_loss.backward()
    model_optimizer.step()
    model_drop_optimizer.step()
    model_weight_decay_optimizer.step()

    # validation predictions and loss
    model.eval()
    model_drop.eval()
    model_weight_decay.eval()
    with torch.no_grad():
        model_y_val_pred = model(X_val_torch)
        model_drop_y_val_pred = model_drop(X_val_torch)
        model_weight_decay_y_val_pred = model_weight_decay(X_val_torch)
        val_losses['model'].append(criterion(model_y_val_pred, y_val_torch))
        val_losses['dropout'].append(criterion(model_drop_y_val_pred, y_val_torch))
        val_losses['weight_decay'].append(criterion(model_weight_decay_y_val_pred, y_val_torch))

    # print out metrics during training
    with torch.no_grad():
        if epoch % 2000 == 0:
            print()
            print(f'Epoch: {epoch}')
            print(f'Train Losses:')
            print(f'Model {model_loss.item()}, Dropout {model_drop_loss.item()}, Weight Decay {model_weight_decay_loss.item()}')
            print(f'Validation Losses:')
            print(f'Model {val_losses["model"][-1]}, Dropout {val_losses["dropout"][-1]}, Weight Decay {val_losses["weight_decay"][-1]}')

print()
print('Training Complete.')
print()


In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 3, 1)
plt.plot(train_losses['model'], label='Train')
plt.plot(val_losses['model'], label='Validation')
plt.title('No Regularization')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(train_losses['dropout'], label='Train')
plt.plot(val_losses['dropout'], label='Validation')
plt.title('Dropout')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(train_losses['weight_decay'], label='Train')
plt.plot(val_losses['weight_decay'], label='Validation')
plt.title('Weight Decay')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.show()